# **Understand the Basics**

**Q1. MapReduce vs Apache Spark**

MapReduce is a traditional framework used for processing large amounts of data. It is comparatively slow because after every processing step, data is stored back on the disk and then read again for the next step. This repeated disk read/write increases execution time.

Apache Spark is a modern Big Data processing framework that works much faster than MapReduce. Its main advantage is in-memory processing, meaning it keeps data in RAM instead of repeatedly using disk storage. This reduces processing time significantly. Spark also provides DataFrames, which help manage structured data efficiently. DataFrames are immutable, so existing data remains unchanged and all modifications create a new DataFrame.

**Q2. Spark DataFrames**

In Spark, data is stored in DataFrames, which are similar to tables containing rows and columns.

Immutability means that once a DataFrame is created, it cannot be modified directly. Any operation such as filtering, cleaning, or transforming data produces a new DataFrame, while the original DataFrame stays unchanged. This helps maintain data consistency and protects raw data from accidental changes.


# **Installing Pyspark and Libraries**

In [ ]:
# 1. Install PySpark
!pip install pyspark

# 2. Import the required Spark libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, min, max, count

# Create the Spark session

In [ ]:
spark = SparkSession.builder \
    .appName("Superstore_Data_Processing") \
    .getOrCreate()

print("Spark Session is Created Successfully")

Spark Session is Created Successfully


# Loading the data

In [ ]:
data = "/content/Superstore_dataset.csv"
df = spark.read.csv(data, header=True, inferSchema=True, escape='"')

print("First 10 Rows")
df.show(10)

First 10 Rows
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|

In [ ]:
print("Data Schema")
df.printSchema()

Data Schema
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



 # **Data Cleaning**

In [ ]:
# 1. Remove duplicate rows
cleaned_df = df.dropDuplicates()

# 2. dropping rows that contains null value
cleaned_df = cleaned_df.dropna()

# 3. count the rows to see if anything was removed
original_count = df.count()
cleaned_count = cleaned_df.count()

print(f"Original Row Count: {original_count}")
print(f"Cleaned Row Count: {cleaned_count}")
print(f"Total Rows Cleaned/Removed: {original_count - cleaned_count}")

Original Row Count: 9994
Cleaned Row Count: 9994
Total Rows Cleaned/Removed: 0


# Filter Data


In [ ]:
cleaned_df = cleaned_df.withColumn("Sales", col("Sales").cast("double"))

filtered_df = cleaned_df \
    .filter(col("Category") == "Furniture") \
    .filter(col("Region") == "West") \
    .filter(col("Sales") >= 100)

# final results
print("Filtered Data")
filtered_df.show(10)

print(f"Total Rows matching our filters: {filtered_df.count()}")

Filtered Data
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID| Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------+------------+--------------------+--------+--------+--------+---------+
|   416|CA-2017-142636| 11/3/2017| 11/7/2017|Standard Class|   KC-16675|  Kimberly Carter|  Corporate|United States|      Seattle|Washington|      98105|  West|FUR-CH-10001891|Furniture|      Chairs|Global Deluxe

# Aggregation and Grouping

In [ ]:
aggregated_df = cleaned_df.groupBy("Region").agg(
    count("*").alias("Total_Orders"),
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sale_Value"),
    max("Sales").alias("Highest_Single_Sale"),
    min("Sales").alias("Lowest_Single_Sale")
)

# results
print("Sales Aggregation by Region ")
aggregated_df.show()

Sales Aggregation by Region 
+-------+------------+-----------------+------------------+-------------------+------------------+
| Region|Total_Orders|      Total_Sales|Average_Sale_Value|Highest_Single_Sale|Lowest_Single_Sale|
+-------+------------+-----------------+------------------+-------------------+------------------+
|  South|        1620|       391721.905|241.80364506172842|           22638.48|             1.167|
|Central|        2323|501239.8907999995|215.77266069737388|           17499.95|             0.444|
|   East|        2848|678781.2400000005|238.33610955056196|          11199.968|             0.852|
|   West|        3203|725457.8245000001|226.49323275054638|           13999.96|              0.99|
+-------+------------+-----------------+------------------+-------------------+------------------+



**Wide Transformations:**
When we used groupBy("Region"), Spark could not process each row independently. Instead, it needed to collect all rows belonging to the same region (like all "West" rows) from different partitions and bring them together. Operations that require data from multiple partitions to be combined are called Wide Transformations.

**Shuffling:**
To make this grouping possible, Spark moves data between different partitions or processors across the network. This transfer of data is called Shuffling. Since moving data over the network takes time and resources, shuffling is usually the slowest and most expensive part of a Spark job.

# Output in csv file

In [ ]:
output = "/content/output"
aggregated_df.write.csv(output, header=True, mode="overwrite")

print("results saved.")

results saved.


Success! Full data pipeline complete and results saved in csv file.